## Normalizasyon nedir?
- Veri setindeki değişkenlerin farklı ölçeklerde olması model performansını olumsuz etkiler.
- Normalizasyon, verileri ortak bir ölçeğe getirerek daha dengeli bir öğrenme sağlar.
- Amaç: Tüm özelliklerim modele eşit katkı yapabilmesini sağlamak
- Örnek: Yaş (0-100) ve Gelir (0-100.000) -> ölçek farkı modeli yanılabilir

## Neden Normalizasyon Gerekir?
- Değişkenlerin ölçü birimleri farklı olabilir (TL,yaş,mesafe,sıcaklık).
- Ölçek farkı, modelin bazı değişkenleri "daha önemli" sanmasına yol açar.
- Normalizasyon, bu farklı giderir ve veriyi karşılaştırabilir hale getirir.
- Modelin öğrenme sürecinde adil ve dengeli bir yapı sağlar.

## Normalizasyonun Temel Mantığı
- Her sütundaki değerler belli bir aralığa dönüştürülür.
- En küçük değer 0, en büyük değer 1 olacak şekilde ölçeklenir.
- Sıralama korunur, sadece ölçek küçülür.


## Normalizasyon Türleri

### Min-Max Normalizayon
- Değerleri [0,1] aralığına çeker.
- Basit, hızlı ve sezgiseldir.
- Aykırı değerlere duyarlıdır.
- (x - min) / (max - min)

### Z-Score (Standardizasyon)
- Ortalama = 0, Standart sapma = 1
- Negatif ve pozitif değerler olabilir.
- Aykırı değerlerden min-max kadar etkilenmez.

### Robust Scaling
- Medyan ve çeyrekler arası fark (IQR) kullanılır.
- Uç değerlere karşı dayanıklıdır.
- Dağılımın merkezine odaklanır.


### Log Dönüşümü
- Çarpık dağılımı daha simetrik hale getirir.
- Büyük değerleri küçültür, küçük farkları belirginleştirir.
- Negatif değerlere uygulanmaz.

# Normalizasyon
- Manuel Normalizasyon
- Kütüphane kullanarak Normalizasyon
- Normalizasyon yapmazsak ne olur?
- Durum çalışması: gerçek veri seti ile normalizasyon çalışması


## Manuel Normalizasyon
Bu bölümde:
- Küçük bir örnek veri seti oluşturalım
- Dört farklı normalizasyon yöntemini formüller ile kendimiz uygulayalım:
  - Min-Max normalizasyon
  - Z-Score standardization
  - Robust Scaling
  - Log Dönüşüm
- Her yöntemin çıktısını değerlendir

In [4]:
# gerekli kütüphaneleri içeri aktar
import numpy as np
import pandas as pd

In [5]:
#veri seti oluşturma
#öğrencilerin boyu, kilosu, gelir durumu
data = {
    "boy": [150,160,170,180,190],
    "kilo": [45,60,70,80,120],
    "gelir": [20000,25000,27000,30000,150000] #son değer outlier
}
df = pd.DataFrame(data)
df

,boy,kilo,gelir
0,150,45,20000
1,160,60,25000
2,170,70,27000
3,180,80,30000
4,190,120,150000


In [6]:
# manuel normalizasyon
# min max normalizasyon
# formül: (X - min) / (max - min)
df_minmax = (df - df.min()) / (df.max() - df.min())
df_minmax

,boy,kilo,gelir
0,0.00,0.000000,0.000000
1,0.25,0.200000,0.038462
2,0.50,0.333333,0.053846
3,0.75,0.466667,0.076923
4,1.00,1.000000,1.000000


In [7]:
# z-score (standardizasyon)
# formül = (X - mean) / std
df_zscore = (df - df.mean()) / df.std()
df_zscore

,boy,kilo,gelir
0,-1.264911,-1.060660,-0.544833
1,-0.632456,-0.530330,-0.455222
2,0.000000,-0.176777,-0.419378
3,0.632456,0.176777,-0.365611
4,1.264911,1.590990,1.785044


In [9]:
# robust scaling
# formül = (X - median) / IQR
# IQR = 3.çeyrek - 1.çeyrek
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1
df_robust = (df - df.median()) / IQR
df_robust

,boy,kilo,gelir
0,-1.0,-1.25,-1.4
1,-0.5,-0.50,-0.4
2,0.0,0.00,0.0
3,0.5,0.50,0.6
4,1.0,2.50,24.6


In [12]:
# log dönüşümü
# formül = log(X + 1)
df_log = np.log(df + 1)
df_log

,boy,kilo,gelir
0,5.017280,3.828641,9.903538
1,5.081404,4.110874,10.126671
2,5.141664,4.262680,10.203629
3,5.198497,4.394449,10.308986
4,5.252273,4.795791,11.918397


In [15]:
summary = pd.DataFrame({
    "orijinal_ortalama": df.mean(),
    "minmax_ortalama": df_minmax.mean(),
    "zscore_ortalama": df_zscore.mean(),
    "robust_ortalama": df_robust.mean(),
    "log_ortalama": df_log.mean()
}).round(3)
summary

,orijinal_ortalama,minmax_ortalama,zscore_ortalama,robust_ortalama,log_ortalama
boy,170.0,0.500,0.0,0.00,5.138
kilo,75.0,0.400,0.0,0.25,4.278
gelir,50400.0,0.234,-0.0,4.68,10.492


## Python Sklearn ile Normalizasyon
Bu bölümde:
- Yeni bir veri seti oluşturalım (mağaza satışları)
- 4 farklı ölçeklendirme yöntemini uygula: min-max, z-score,robust ve log dönüşümü
- Sonuçları karşılaştıralım


In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

In [18]:
# örnek veri seti oluştur
# farklı mağazalara ait satış ve müşteri bilgileri
data = {
    "satis_tutari": [2000,5000,7000,10000,120000],
    "musteri_sayisi": [50,65,80,100,300],
    "urun_cesidi": [10,20,25,30,60],
    "iade_orani": [0.02,0.03,0.05,0.04,0.5]
}
df = pd.DataFrame(data)
df

,satis_tutari,musteri_sayisi,urun_cesidi,iade_orani
0,2000,50,10,0.02
1,5000,65,20,0.03
2,7000,80,25,0.05
3,10000,100,30,0.04
4,120000,300,60,0.50


In [19]:
# min max normalizasyon
# formül: (X- Xmin) / (Xmax- Xmin)
scaler_minmax = MinMaxScaler()
df_minmax = pd.DataFrame(scaler_minmax.fit_transform(df),columns=df.columns)
df_minmax

,satis_tutari,musteri_sayisi,urun_cesidi,iade_orani
0,0.000000,0.00,0.0,0.000000
1,0.025424,0.06,0.2,0.020833
2,0.042373,0.12,0.3,0.062500
3,0.067797,0.20,0.4,0.041667
4,1.000000,1.00,1.0,1.000000


In [20]:
# z score
# formul = (X - mean) / std
scaler_standard = StandardScaler()
df_zscore = pd.DataFrame(scaler_standard.fit_transform(df),columns=df.columns)
df_zscore

,satis_tutari,musteri_sayisi,urun_cesidi,iade_orani
0,-0.586761,-0.750000,-1.127443,-0.579808
1,-0.521078,-0.586957,-0.534052,-0.526122
2,-0.477290,-0.423913,-0.237356,-0.418750
3,-0.411608,-0.206522,0.059339,-0.472436
4,1.996738,1.967391,1.839512,1.997116


In [21]:
# robust scaler
# formül = (X - median) / IQR
scaler_robust = RobustScaler()
df_robust = pd.DataFrame(scaler_robust.fit_transform(df), columns = df.columns)
df_robust

,satis_tutari,musteri_sayisi,urun_cesidi,iade_orani
0,-1.0,-0.857143,-1.5,-1.0
1,-0.4,-0.428571,-0.5,-0.5
2,0.0,0.000000,0.0,0.5
3,0.6,0.571429,0.5,0.0
4,22.6,6.285714,3.5,23.0


In [24]:
# log dönüşümü
# formül = log(x+1)
df_log = np.log(df+1)
df_log

,satis_tutari,musteri_sayisi,urun_cesidi,iade_orani
0,7.601402,3.931826,2.397895,0.019803
1,8.517393,4.189655,3.044522,0.029559
2,8.853808,4.394449,3.258097,0.048790
3,9.210440,4.615121,3.433987,0.039221
4,11.695255,5.707110,4.110874,0.405465


In [25]:
# ölçeklendirme sonuçlarının özet tablosu: mean
summary = pd.DataFrame({
    "orijinal_ortalama": df.mean(),
    "min_max_ortalama": df_minmax.mean(),
    "z_score_ortalama": df_zscore.mean(),
    "robust_ortalama": df_robust.mean(),
    "log_ortalama": df_log.mean()
}).round(3)
summary


,orijinal_ortalama,minmax_ortalama,zscore_ortalama,robust_ortalama,log_ortalama
satis_tutari,28800.000,0.227,0.0,4.360,9.176
musteri_sayisi,119.000,0.276,0.0,1.114,4.568
urun_cesidi,29.000,0.380,0.0,0.400,3.249
iade_orani,0.128,0.225,0.0,4.400,0.109


In [26]:
# ölçeklendirme sonuçlarının özet tablosu: mean
summary = pd.DataFrame({
    "orijinal_std": df.std(),
    "min_max_std": df_minmax.std(),
    "z_score_std": df_zscore.std(),
    "robust_std": df_robust.std(),
    "log_std": df_log.std()
}).round(3)
summary


,orijinal_std,min_max_std,z_score_std,robust_std,log_std
satis_tutari,51065.644,0.433,1.118,10.213,1.530
musteri_sayisi,102.859,0.411,1.118,2.939,0.685
urun_cesidi,18.841,0.377,1.118,1.884,0.621
iade_orani,0.208,0.434,1.118,10.413,0.166
